# Batch Simulation Runner

This notebook finds generated scenario/workload files, lets you choose combinations, and runs the simulator for every pair.

In [1]:
# Install dependencies required for this notebook
%pip install ipywidgets

Note: you may need to restart the kernel to use updated packages.


In [17]:
from pathlib import Path
from IPython.display import display, Markdown
import ipywidgets as widgets
import itertools
import subprocess
import shlex
import json
import datetime

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
BUILD_BIN_DIR = PROJECT_ROOT / "build" / "bin"

def find_sim_executable():
    candidates = []
    for ext in ("", ".exe"):
        candidates.extend(BUILD_BIN_DIR.rglob(f"scheduler_sim{ext}"))

    unique = []
    seen = set()
    for cand in candidates:
        if cand in seen:
            continue
        seen.add(cand)
        unique.append(cand)

    def sort_key(path: Path):
        try:
            rel = path.relative_to(BUILD_BIN_DIR)
            depth = len(rel.parts)
        except ValueError:
            depth = 999
        return (depth, str(path))

    for path in sorted(unique, key=sort_key):
        if path.is_file():
            return path

    return BUILD_BIN_DIR / "scheduler_sim"

SIM_EXECUTABLE = find_sim_executable()
DEFAULT_RESULTS_ROOT = (SIM_EXECUTABLE.parent if SIM_EXECUTABLE.exists() else BUILD_BIN_DIR) / "results"
GEN_SCENARIO_DIR = PROJECT_ROOT / "data" / "scenarios" / "generated"
GEN_WORKLOAD_DIR = PROJECT_ROOT / "data" / "workloads" / "generated"

In [18]:
def discover_files(base_dir: Path, suffix: str = '.yml'):
    if not base_dir.exists():
        return []
    return sorted([p for p in base_dir.rglob(f'*{suffix}') if p.is_file()])

scenarios = discover_files(GEN_SCENARIO_DIR) or discover_files(PROJECT_ROOT / 'data' / 'scenarios', '.yml')
workloads = discover_files(GEN_WORKLOAD_DIR) or discover_files(PROJECT_ROOT / 'data' / 'workloads', '.yml')

def format_rel(path: Path) -> str:
    try:
        return str(path.relative_to(PROJECT_ROOT))
    except ValueError:
        return str(path)

scenario_options = [(format_rel(p), str(p)) for p in scenarios]
workload_options = [(format_rel(p), str(p)) for p in workloads]

scenario_select = widgets.SelectMultiple(options=scenario_options, description='Scenarios', layout=widgets.Layout(width='45%', height='200px'))
workload_select = widgets.SelectMultiple(options=workload_options, description='Workloads', layout=widgets.Layout(width='45%', height='200px'))

layout = widgets.HBox([scenario_select, workload_select])
display(Markdown('## Select scenarios and workloads'))
display(layout)

## Select scenarios and workloads

In [19]:
results_subdir = widgets.Text(value=datetime.datetime.now().strftime('batch_%Y%m%d_%H%M%S'), description='Results subdir')
out_file_name = widgets.Text(value='trace.json', description='Output file')
duration_int = widgets.IntText(value=10000, description='Duration (ms)')
log_stdout_level = widgets.Text(value='info', description='Stdout log level')
log_file_level = widgets.Text(value='debug', description='File log level')
log_file_name = widgets.Text(value='logs/batch.log', description='Log file')
override_verbose = widgets.Checkbox(value=False, description='Verbose flag')

policy_options = [
    ("FCFS", "fcfs"),
    ("SJF", "sjf"),
    ("Priority", "priority"),
    ("RR", "rr"),
    ("MLFQ", "mlfq"),
    ("EDF", "edf"),
    ("Linux/CFS", "linux"),
    ("MLQ", "mlq"),
    ("POSIX FIFO", "posix_rt"),
    ("POSIX RR", "sched_rr"),
    ("Priority Aging", "priority_based"),
    ("Proportional", "proportional"),
    ("RMS", "rms"),
    ("Windows", "windows"),
]

default_policies = {"fcfs", "mlfq"}
policy_checks = [widgets.Checkbox(value=val in default_policies, description=label) for label, val in policy_options]
policy_columns = [widgets.VBox(policy_checks[::2]), widgets.VBox(policy_checks[1::2])]
policies_widget = widgets.VBox([widgets.Label('Policies'), widgets.HBox(policy_columns)])

def selected_policies():
    return [val for (label, val), cb in zip(policy_options, policy_checks) if cb.value]

config_box = widgets.VBox([
    results_subdir,
    out_file_name,
    duration_int,
    log_stdout_level,
    log_file_level,
    log_file_name,
    override_verbose,
    policies_widget,
])
display(Markdown('## Simulation configuration'))
display(config_box)


## Simulation configuration

In [ ]:
run_button = widgets.Button(description='Run batch', button_style='success')
run_log = widgets.Output(layout=widgets.Layout(width='100%', height='300px', overflow='auto'))


from pathlib import PurePath

def run_batch(_):
    run_log.clear_output()
    with run_log:
        selected_scenarios = [Path(p) for p in scenario_select.value]
        selected_workloads = [Path(p) for p in workload_select.value]
        if not selected_scenarios:
            print('No scenarios selected.')
            return
        if not selected_workloads:
            print('No workloads selected.')
            return
        policies = selected_policies()
        if not policies:
            print('No scheduling policies selected.')
            return
        if not SIM_EXECUTABLE.exists():
            print(f'Simulator executable not found at {SIM_EXECUTABLE}')
            return
        for scenario_path, workload_path in itertools.product(selected_scenarios, selected_workloads):
            rel_scenario = format_rel(scenario_path)
            rel_workload = format_rel(workload_path)
            subdir = PurePath(results_subdir.value) / scenario_path.stem / workload_path.stem
            cmd = [
                str(SIM_EXECUTABLE),
                '--scenario_file', str(scenario_path),
                '--workload_file', str(workload_path),
                '--scheduler_policy', ','.join(policies),
                '--duration', str(duration_int.value),
                '--out_file', str(DEFAULT_RESULTS_ROOT / 'trace.json'),
                '--log_stdout_level', log_stdout_level.value,
                '--log_file_level', log_file_level.value,
                '--log_file', log_file_name.value,
                '--results_subdir', str(subdir),
            ]
            if override_verbose.value:
                cmd.append('--verbose')
            print('Running:', ' '.join(shlex.quote(part) for part in cmd))
            try:
                subprocess.run(cmd, check=True)
            except subprocess.CalledProcessError as exc:
                print(f'Run failed for scenario={rel_scenario} workload={rel_workload}: {exc}')

run_button.on_click(run_batch)
display(widgets.VBox([run_button, run_log]))
